# HEADS — Single-Notebook ML Pipeline (Transformer AE + GraphSAGE + XGBoost + SMOTE)

✅ This notebook is **self-contained** (no `src/` package needed).

## Fix for your error
You saw:
`RuntimeError: Not enough events per actor to build sequences...`

That happens when most actors (`src_ip`) have fewer events than `seq_len`.
This notebook fixes it by:
- **Auto-adapting `SEQ_LEN`** based on the data
- Building sequences only for actors with enough events
- If still insufficient, **fallback to global sequences** (sorted by timestamp)

---
## Input
Set `DATA_PATH` to your CSV (default: `data/raw/cybersecurity.csv`).


In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score
)

from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Torch Geometric (GraphSAGE)
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv


In [ ]:
# -----------------------------
# CONFIG (edit as needed)
# -----------------------------
DATA_PATH = Path("data/raw/cybersecurity.csv")

# If actors have few events, keep this small (the notebook will auto-adjust anyway)
SEQ_LEN = 20

# Training hyperparams (keep small for laptop)
AE_EPOCHS = 5
AE_BATCH = 256
AE_LR = 3e-4
AE_D_MODEL = 64
AE_NHEAD = 4
AE_LAYERS = 2

GNN_EPOCHS = 5
GNN_LR = 1e-3
GNN_HIDDEN = 64
GNN_LAYERS = 2

XGB_N_EST = 400
XGB_MAX_DEPTH = 5
XGB_LR = 0.05

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


In [ ]:
REQUIRED_COLS = [
    "timestamp","src_ip","dst_ip","src_port","dst_port","protocol",
    "bytes_sent","bytes_received","user_agent","url","is_internal_traffic"
]

def load_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    for c in ["src_port","dst_port","bytes_sent","bytes_received"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

    df["protocol"] = df["protocol"].astype(str).fillna("UNK")
    df["user_agent"] = df["user_agent"].astype(str).fillna("UNK")
    df["url"] = df["url"].astype(str).replace("nan","").fillna("")
    df["is_internal_traffic"] = df["is_internal_traffic"].astype(bool)

    if "label" in df.columns:
        df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)
    if "attack_type" in df.columns:
        df["attack_type"] = df["attack_type"].astype(str).fillna("benign")

    df = df.sort_values(["src_ip","timestamp"], kind="mergesort").reset_index(drop=True)
    return df

assert DATA_PATH.exists(), f"CSV not found: {DATA_PATH.resolve()}"
df = load_csv(DATA_PATH)
df.head()


In [ ]:
def extract_url_host(url: str) -> str:
    if not url:
        return ""
    m = re.match(r"^https?://([^/]+)/?", url)
    return m.group(1).lower() if m else ""

def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["hour"] = out["timestamp"].dt.hour.fillna(0).astype(int)
    out["dow"] = out["timestamp"].dt.dayofweek.fillna(0).astype(int)
    out["url_host"] = out["url"].apply(extract_url_host)
    out["bytes_total"] = (out["bytes_sent"] + out["bytes_received"]).astype(int)
    out["is_web"] = out["dst_port"].isin([80,443]).astype(int)
    out["is_internal_traffic"] = out["is_internal_traffic"].astype(int)
    return out

df = add_derived_features(df)

print("rows:", len(df))
print("unique src_ip:", df["src_ip"].nunique())
if "label" in df.columns:
    print("attack rate:", float((df["label"]==1).mean()))


In [ ]:
# -----------------------------
# Transformer Autoencoder
# -----------------------------
class SeqDataset(Dataset):
    def __init__(self, X_seq: np.ndarray):
        self.X = X_seq.astype(np.float32)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i):
        return torch.from_numpy(self.X[i])

class TransformerAutoencoder(nn.Module):
    def __init__(self, feat_dim: int, d_model: int, nhead: int, num_layers: int, dropout: float = 0.1):
        super().__init__()
        self.proj_in = nn.Linear(feat_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        dec_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True)
        self.decoder = nn.TransformerEncoder(dec_layer, num_layers=max(1, num_layers))
        self.proj_out = nn.Linear(d_model, feat_dim)

    def forward(self, x):
        z = self.proj_in(x)
        z = self.encoder(z)
        z = self.decoder(z)
        return self.proj_out(z)

def build_sequences_adaptive(df: pd.DataFrame, feature_cols: list[str], seq_len: int, group_col: str = "src_ip"):
    """Build sequences per actor if possible; otherwise fall back to global sequences."""
    counts = df.groupby(group_col).size()
    max_events = int(counts.max()) if len(counts) else 0

    eff_len = min(seq_len, max_events)
    if eff_len < 3:
        eff_len = 3

    X_list = []
    idx_list = []

    # Actor-wise sequences
    for _, g in df.groupby(group_col, sort=False):
        g = g.sort_values("timestamp")
        if len(g) < eff_len:
            continue
        X = g[feature_cols].to_numpy(dtype=float)
        for i in range(eff_len-1, len(g)):
            X_list.append(X[i-eff_len+1:i+1])
            idx_list.append(g.index[i])

    if len(X_list) >= 50:
        return np.stack(X_list), np.array(idx_list, dtype=int), eff_len, "per-actor"

    # Fallback: global sequences
    g = df.sort_values("timestamp")
    X = g[feature_cols].to_numpy(dtype=float)
    if len(g) < eff_len:
        raise RuntimeError(f"Not enough rows ({len(g)}) even for effective seq_len={eff_len}. Reduce SEQ_LEN.")
    X_list = []
    idx_list = []
    for i in range(eff_len-1, len(g)):
        X_list.append(X[i-eff_len+1:i+1])
        idx_list.append(g.index[i])
    return np.stack(X_list), np.array(idx_list, dtype=int), eff_len, "global-fallback"

def train_autoencoder(X_seq: np.ndarray, feat_dim: int):
    model = TransformerAutoencoder(feat_dim, AE_D_MODEL, AE_NHEAD, AE_LAYERS).to(DEVICE)
    dl = DataLoader(SeqDataset(X_seq), batch_size=AE_BATCH, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=AE_LR)
    loss_fn = nn.MSELoss()

    model.train()
    for ep in range(AE_EPOCHS):
        total = 0.0
        for x in dl:
            x = x.to(DEVICE)
            x_hat = model(x)
            loss = loss_fn(x_hat, x)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += float(loss.item()) * x.size(0)
        print(f"[AE] epoch {ep+1}/{AE_EPOCHS} loss={total/len(dl.dataset):.6f}")
    return model

@torch.no_grad()
def score_autoencoder(model: nn.Module, X_seq: np.ndarray):
    model.eval()
    dl = DataLoader(SeqDataset(X_seq), batch_size=512, shuffle=False)
    scores=[]
    for x in dl:
        x = x.to(DEVICE)
        x_hat = model(x)
        err = torch.mean((x_hat - x)**2, dim=(1,2))
        scores.append(err.detach().cpu().numpy())
    return np.concatenate(scores)

ae_cols = ["src_port","dst_port","bytes_sent","bytes_received","bytes_total","hour","dow","is_web","is_internal_traffic"]
X_seq, idx_last, eff_len, mode = build_sequences_adaptive(df, ae_cols, SEQ_LEN, group_col="src_ip")
print("sequence_mode:", mode, "| effective_seq_len:", eff_len, "| sequences:", X_seq.shape)

ae = train_autoencoder(X_seq, feat_dim=len(ae_cols))
temporal_scores = score_autoencoder(ae, X_seq)

df["temporal_score"] = 0.0
df.loc[idx_last, "temporal_score"] = temporal_scores
df["temporal_score"].describe()


In [ ]:
# -----------------------------
# GraphSAGE relational model
# -----------------------------
def build_graph(df: pd.DataFrame):
    node_map = {}
    node_types = []
    edges = []

    def get_node(t: str, key: str):
        k=(t,key)
        if k in node_map:
            return node_map[k]
        nid=len(node_map)
        node_map[k]=nid
        node_types.append(t)
        return nid

    for _, r in df.iterrows():
        a = get_node("actor", str(r["src_ip"]))
        b = get_node("resource", str(r["dst_ip"]))
        edges.append((a,b))
        h = str(r.get("url_host",""))
        if h and h != "nan":
            hh = get_node("host", h)
            edges.append((a,hh))
        p = get_node("proto", str(r["protocol"]))
        edges.append((a,p))

    edge_index = np.array(edges, dtype=np.int64).T
    num_nodes = len(node_map)

    types_sorted = sorted(set(node_types))
    type_to_id = {t:i for i,t in enumerate(types_sorted)}
    type_ids = np.array([type_to_id[t] for t in node_types], dtype=np.float32)

    deg = np.zeros(num_nodes, dtype=np.float32)
    for u,v in edges:
        deg[u]+=1; deg[v]+=1
    x = np.stack([type_ids, np.log1p(deg)], axis=1)

    data = Data(
        x=torch.tensor(x, dtype=torch.float32),
        edge_index=torch.tensor(edge_index, dtype=torch.long),
        num_nodes=num_nodes
    )
    return data, node_map

class GraphSAGEEncoder(nn.Module):
    def __init__(self, in_dim: int, hidden: int, layers: int):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(in_dim, hidden))
        for _ in range(layers-1):
            self.convs.append(SAGEConv(hidden, hidden))
    def forward(self, x, edge_index):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = torch.relu(x)
        return x

def negative_sampling(num_nodes: int, num_neg: int):
    src = torch.randint(0, num_nodes, (num_neg,), device=DEVICE)
    dst = torch.randint(0, num_nodes, (num_neg,), device=DEVICE)
    return torch.stack([src,dst], dim=0)

def train_graphsage(data: Data):
    data = data.to(DEVICE)
    model = GraphSAGEEncoder(data.x.size(1), GNN_HIDDEN, GNN_LAYERS).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=GNN_LR)

    E = data.edge_index.size(1)
    for ep in range(GNN_EPOCHS):
        model.train()
        z = model(data.x, data.edge_index)

        src, dst = data.edge_index
        pos_score = (z[src]*z[dst]).sum(dim=1)

        neg_e = negative_sampling(data.num_nodes, E)
        ns, nd = neg_e
        neg_score = (z[ns]*z[nd]).sum(dim=1)

        loss = -torch.mean(torch.log(torch.sigmoid(pos_score)+1e-9)) - torch.mean(torch.log(torch.sigmoid(-neg_score)+1e-9))
        opt.zero_grad()
        loss.backward()
        opt.step()
        print(f"[GNN] epoch {ep+1}/{GNN_EPOCHS} loss={float(loss):.6f}")
    return model

@torch.no_grad()
def edge_anomaly_scores(model, data: Data, edge_pairs: np.ndarray):
    data = data.to(DEVICE)
    model.eval()
    z = model(data.x, data.edge_index)
    edge_index = torch.tensor(edge_pairs.T, dtype=torch.long, device=DEVICE)
    src, dst = edge_index
    score = (z[src]*z[dst]).sum(dim=1)
    return (-torch.log(torch.sigmoid(score)+1e-9)).detach().cpu().numpy()

graph_data, node_map = build_graph(df)
gnn = train_graphsage(graph_data)

pairs=[]
for _, r in df.iterrows():
    a = node_map[("actor", str(r["src_ip"]))]
    b = node_map[("resource", str(r["dst_ip"]))]
    pairs.append((a,b))
pairs = np.array(pairs, dtype=np.int64)

df["relational_score"] = edge_anomaly_scores(gnn, graph_data, pairs)
df["relational_score"].describe()


In [ ]:
# -----------------------------
# XGBoost fusion + SMOTE
# -----------------------------
from xgboost import XGBClassifier

class TabularFeaturizer:
    def __init__(self, text_dim: int = 64):
        self.ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        self.hv = HashingVectorizer(n_features=text_dim, alternate_sign=False, norm=None)
        self.fitted=False

    def fit(self, df: pd.DataFrame):
        cat = df[["protocol","url_host","is_internal_traffic"]].astype(str)
        self.ohe.fit(cat)
        self.fitted=True
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        if not self.fitted:
            raise RuntimeError("fit first")
        num = df[["src_port","dst_port","bytes_sent","bytes_received","bytes_total","hour","dow","is_web"]].to_numpy(float)
        cat = df[["protocol","url_host","is_internal_traffic"]].astype(str)
        cat_mat = self.ohe.transform(cat)
        text = (df["url"].fillna("") + " " + df["user_agent"].fillna("")).astype(str).tolist()
        text_mat = self.hv.transform(text).toarray().astype(float)
        return np.hstack([num, cat_mat, text_mat])

if "label" not in df.columns:
    print("No label column → skipping supervised fusion model. You still have temporal_score + relational_score.")
else:
    feat = TabularFeaturizer(text_dim=64).fit(df)
    X_tab = feat.transform(df)
    X = np.hstack([X_tab, df[["temporal_score","relational_score"]].to_numpy(float)])
    y = df["label"].astype(int).to_numpy()

    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

    k = min(5, max(1, int((y_tr==1).sum()-1)))
    sm = SMOTE(random_state=42, k_neighbors=k)
    X_tr2, y_tr2 = sm.fit_resample(X_tr, y_tr)

    xgb = XGBClassifier(
        n_estimators=XGB_N_EST,
        max_depth=XGB_MAX_DEPTH,
        learning_rate=XGB_LR,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="auc",
        n_jobs=-1,
        random_state=42
    )
    xgb.fit(X_tr2, y_tr2)

    y_score = xgb.predict_proba(X_va)[:,1]
    y_pred = (y_score >= 0.5).astype(int)

    print("ROC-AUC:", roc_auc_score(y_va, y_score))
    print("PR-AUC :", average_precision_score(y_va, y_score))
    print(classification_report(y_va, y_pred, zero_division=0))

    df["xgb_proba"] = xgb.predict_proba(X)[:,1]
    df["prediction"] = (df["xgb_proba"] >= 0.5).astype(int)

df.head()


In [ ]:
# -----------------------------
# Export results
# -----------------------------
out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / "scored_events.csv"
df.to_csv(out_csv, index=False)
out_csv


## You now have a single .ipynb file
- It **fixes the seq_len issue automatically**.
- It exports: `data/processed/scored_events.csv`

If you want, I can also embed the Streamlit dashboard code into cells (still one notebook),
but for real production deployment it’s better as a separate `streamlit_app/` folder.
